# Experiment B - Smoothed xT Matrix Clustering

## 1. Markdown Introduction

This notebook performs Experiment B for unsupervised clustering of team-season tactical styles using the smoothed team-season xT created distribution.

Each row represents one team in one season. Each team-season is represented by a 192-zone spatial vector from a 16 by 12 pitch grid, where `zone = y_bin * 16 + x_bin`. The goal is to cluster team-seasons based on where they create positive xT after numerical spatial smoothing.

Smoothing can reduce noise from isolated high-value zones, which may make clusters more stable and easier to interpret. However, too much smoothing can also blur real tactical differences. Experiment B should therefore be compared with Experiment A, the raw xT matrix baseline, when those outputs are available.

These clusters describe possession threat creation patterns. They should not be overread as complete tactical identities: this feature set does not directly measure pressing, defensive block height, rest defense, or counterpressing. Treat cluster labels as analytical hypotheses rather than ground-truth tactical categories.


## 2. Imports and Configuration

This setup cell defines the Experiment B contract: one processed smoothed feature CSV, one output folder, a 16 by 12 pitch grid, PCA variance retention, and KMeans settings. The notebook does not retrain xT, parse raw StatsBomb files, recalculate action-level xT, rebuild team-season matrices, or use PNG heatmaps as machine learning input.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This setup cell defines the Experiment B contract: one input CSV, one output folder,
# a 16 by 12 pitch grid, PCA variance retention, and KMeans cluster settings.
# The formula N_ZONES = GRID_L * GRID_W gives 16 * 12 = 192 spatial features.
# --- End learning comments ---

from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import pairwise_distances_argmin_min


INPUT_FILE = Path("outputs/team_season_features_smoothed_created_xt_distribution.csv")
OUTPUT_DIR = Path("outputs/experiment_B_smoothed/")

EXPERIMENT_A_CLUSTERED_FILE = Path("outputs/experiment_A_raw/clustered_team_seasons_raw.csv")
EXPERIMENT_A_SUMMARY_FILE = Path("outputs/experiment_A_raw/cluster_summary_raw.csv")
EXPERIMENT_A_REPRESENTATIVES_FILE = Path("outputs/experiment_A_raw/cluster_representatives_raw.csv")

EXPERIMENT_NAME = "Experiment B - Smoothed xT Matrix Clustering"
FEATURE_PREFIX_USED = "smooth_created_z"

GRID_L = 16
GRID_W = 12
N_ZONES = GRID_L * GRID_W

MIN_MATCH_COUNT = 5
K_RANGE = range(3, 11)
FINAL_K = 5
RANDOM_STATE = 42
PCA_VARIANCE_TARGET = 0.85

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Feature prefix: {FEATURE_PREFIX_USED}")


## 3. Load and Validate Data

This section loads only the processed smoothed team-season feature CSV. It validates that the 192 clustering inputs are the smoothed xT distribution columns, not raw event data and not image files.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell validates that the input file is a processed smoothed feature matrix, not raw event data.
# Feature detection requires smooth_created_z000...smooth_created_z191.
# The clustering algorithm later uses only these 192 zone columns.
# --- End learning comments ---

if not INPUT_FILE.exists():
    similar_files = sorted(str(path) for path in INPUT_FILE.parent.glob("*team_season*"))
    similar_preview = "\n".join(similar_files[:30]) if similar_files else "No nearby team-season files found."
    raise FileNotFoundError(
        f"Expected input file not found: {INPUT_FILE}\n\n"
        "Experiment B should load the processed smoothed created xT distribution CSV with "
        "columns smooth_created_z000 through smooth_created_z191.\n"
        "Create or copy that processed file before running this notebook.\n\n"
        f"Nearby team-season files found:\n{similar_preview}"
    )

data = pd.read_csv(INPUT_FILE)

print(f"Dataset shape: {data.shape}")
print(f"Number of rows: {len(data):,}")
print(f"Number of columns: {data.shape[1]:,}")

if "team_name" in data.columns:
    print(f"Unique teams: {data['team_name'].nunique(dropna=True):,}")
elif "team_id" in data.columns:
    print(f"Unique teams: {data['team_id'].nunique(dropna=True):,}")
if "season_name" in data.columns:
    print(f"Seasons: {data['season_name'].nunique(dropna=True):,}")
elif "season_id" in data.columns:
    print(f"Seasons: {data['season_id'].nunique(dropna=True):,}")
if "competition_name" in data.columns:
    print(f"Competitions: {data['competition_name'].nunique(dropna=True):,}")
elif "competition_id" in data.columns:
    print(f"Competitions: {data['competition_id'].nunique(dropna=True):,}")


def zone_number_from_column(column_name, prefix=FEATURE_PREFIX_USED):
    match = re.search(rf"^{re.escape(prefix)}(\d+)$", column_name)
    if match is None:
        return None
    return int(match.group(1))


feature_pairs = []
for column in data.columns:
    zone = zone_number_from_column(column, FEATURE_PREFIX_USED)
    if zone is not None:
        feature_pairs.append((zone, column))

if not feature_pairs:
    raise ValueError(
        "No spatial feature columns found. Expected columns starting with "
        f"'{FEATURE_PREFIX_USED}'. Experiment B must use the smoothed xT distribution features."
    )

feature_pairs = sorted(feature_pairs)
feature_zone_ids = [zone for zone, _ in feature_pairs]
feature_columns = [column for _, column in feature_pairs]
metadata_columns = [column for column in data.columns if column not in feature_columns]

print(f"Feature prefix used: {FEATURE_PREFIX_USED}")
print(f"Number of feature columns found: {len(feature_columns)}")

if len(feature_columns) != N_ZONES:
    raise ValueError(f"Expected exactly {N_ZONES} feature columns, found {len(feature_columns)}.")

expected_zones = list(range(N_ZONES))
if feature_zone_ids != expected_zones:
    raise ValueError(
        "Feature columns do not cover exactly z000 to z191 in order. "
        f"First detected zones: {feature_zone_ids[:10]}; last detected zones: {feature_zone_ids[-10:]}"
    )

feature_numeric = data[feature_columns].apply(pd.to_numeric, errors="coerce")
coerced_to_missing = feature_numeric.isna() & data[feature_columns].notna()
non_numeric_count = int(coerced_to_missing.sum().sum())
if non_numeric_count:
    raise ValueError(f"Found {non_numeric_count:,} non-numeric values in feature columns.")
data[feature_columns] = feature_numeric

all_missing_features = data[feature_columns].columns[data[feature_columns].isna().all()].tolist()
if all_missing_features:
    raise ValueError(f"These feature columns are entirely missing: {all_missing_features[:10]}")

id_duplicate_keys = ["competition_id", "season_id", "team_id"]
name_duplicate_keys = ["competition_name", "season_name", "team_name"]
if all(column in data.columns for column in id_duplicate_keys):
    duplicate_key_columns = id_duplicate_keys
elif all(column in data.columns for column in name_duplicate_keys):
    duplicate_key_columns = name_duplicate_keys
else:
    duplicate_key_columns = []

if duplicate_key_columns:
    duplicate_rows = data.duplicated(subset=duplicate_key_columns, keep=False)
    print(f"Duplicate team-season rows using {duplicate_key_columns}: {int(duplicate_rows.sum()):,}")
else:
    print("No complete team-season key was available for duplicate-row checking.")

metadata_missing = data[metadata_columns].isna().sum().sort_values(ascending=False)
metadata_missing = metadata_missing[metadata_missing > 0]
feature_missing_counts = data[feature_columns].isna().sum()

print(f"Metadata columns: {metadata_columns}")
print(f"Missing values in metadata columns: {int(metadata_missing.sum()):,}")
if not metadata_missing.empty:
    display(metadata_missing.head(20).rename("missing_values").to_frame())

print(f"Missing values in feature columns: {int(feature_missing_counts.sum()):,}")
print(f"Feature columns entirely missing: {len(all_missing_features)}")
print(f"First few feature columns: {feature_columns[:8]}")

print("Feature value summary:")
display(data[feature_columns].describe().T[["mean", "std", "min", "max"]].head(12))
display(data[metadata_columns + feature_columns[:5]].head())


## 4. Data Filtering

This section removes rows that would make clustering unreliable: team-seasons with too few matches, no smoothed xT signal, or missing spatial features. The filtering report is saved so the sample definition is explicit.

Filtering order:

1. If `match_count` exists, remove rows where `match_count < MIN_MATCH_COUNT`.
2. Remove rows where all selected feature values sum to 0.
3. Remove rows with missing values in the selected feature columns.

The filtering audit is saved to:

```text
outputs/experiment_B_smoothed/filtering_report_smoothed.csv
```


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell filters the sample before clustering.
# The main row-level formula is s_i = sum_j X_ij.
# Rows with too few matches, a near-zero feature sum, or missing zone values are removed.
# --- End learning comments ---

initial_row_count = len(data)
filtered = data.copy()

removed_low_match_count = 0
removed_zero_feature_sum = 0
removed_missing_features = 0

if "match_count" in filtered.columns:
    filtered["match_count"] = pd.to_numeric(filtered["match_count"], errors="coerce")
    low_match_mask = filtered["match_count"].fillna(-np.inf) < MIN_MATCH_COUNT
    removed_low_match_count = int(low_match_mask.sum())
    filtered = filtered.loc[~low_match_mask].copy()
else:
    print("Column 'match_count' not found, so the minimum-match filter is skipped.")

feature_sum = filtered[feature_columns].sum(axis=1, skipna=False)
zero_feature_mask = np.isclose(feature_sum.fillna(np.nan), 0)
removed_zero_feature_sum = int(zero_feature_mask.sum())
filtered = filtered.loc[~zero_feature_mask].copy()

missing_feature_mask = filtered[feature_columns].isna().any(axis=1)
removed_missing_features = int(missing_feature_mask.sum())
filtered = filtered.loc[~missing_feature_mask].copy()

final_row_count = len(filtered)

filtering_report = pd.DataFrame(
    [
        {"step": "initial_rows", "row_count": initial_row_count},
        {"step": "removed_low_match_count", "row_count": removed_low_match_count},
        {"step": "removed_zero_feature_sum", "row_count": removed_zero_feature_sum},
        {"step": "removed_missing_feature_values", "row_count": removed_missing_features},
        {"step": "final_rows", "row_count": final_row_count},
    ]
)
filtering_report.to_csv(OUTPUT_DIR / "filtering_report_smoothed.csv", index=False)

print(f"Initial row count: {initial_row_count:,}")
print(f"Rows removed because of low match_count: {removed_low_match_count:,}")
print(f"Rows removed because feature sum is zero: {removed_zero_feature_sum:,}")
print(f"Rows removed because of missing feature values: {removed_missing_features:,}")
print(f"Final row count: {final_row_count:,}")
print(f"Saved filtering report: {OUTPUT_DIR / 'filtering_report_smoothed.csv'}")

display(filtering_report)

if final_row_count <= FINAL_K:
    raise ValueError(
        f"Not enough filtered samples ({final_row_count}) for FINAL_K={FINAL_K}. "
        "Lower FINAL_K or relax filtering."
    )


## 5. Feature Matrix Preparation

Only the 192 smoothed xT distribution columns are used for clustering. Because the input is a sum-normalized smoothed positive xT distribution, the model clusters spatial style rather than total attacking volume.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell separates metadata from the numerical feature matrix X.
# Each row of X should be a spatial distribution with sum_j X_ij approximately equal to 1.
# That makes the clustering compare spatial preference rather than total xT volume.
# --- End learning comments ---

metadata_filtered = filtered[metadata_columns].reset_index(drop=True)
X = filtered[feature_columns].reset_index(drop=True)

row_sums = X.sum(axis=1)
nonzero_row_sums = row_sums.loc[~np.isclose(row_sums, 0)]
max_abs_deviation_from_one = float((nonzero_row_sums - 1).abs().max()) if len(nonzero_row_sums) else np.nan

print(f"Feature matrix shape: {X.shape}")
print(f"Min row sum: {row_sums.min():.6f}")
print(f"Max row sum: {row_sums.max():.6f}")
print(f"Mean row sum: {row_sums.mean():.6f}")
print(f"Maximum absolute deviation from 1 for nonzero rows: {max_abs_deviation_from_one:.6f}")

if max_abs_deviation_from_one > 0.05:
    print(
        "Warning: row sums deviate from 1 by more than 0.05. "
        "Confirm that the input file is a distribution dataset."
    )

## 6. Standardization

Even though the rows are already sum-normalized, StandardScaler helps KMeans because different pitch zones may have different cross-team variances. This keeps higher-variance zones from dominating Euclidean distance only because of scale.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell standardizes every zone feature with z = (x - mean) / std.
# KMeans is distance-based, so unscaled high-variance zones could dominate Euclidean distance.
# --- End learning comments ---

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Mean of scaled features, average absolute value: {np.abs(X_scaled.mean(axis=0)).mean():.6f}")
print(f"Std of scaled features, average value: {X_scaled.std(axis=0).mean():.6f}")

## 7. PCA Dimensionality Reduction

PCA reduces the 192-zone smoothed matrix into lower-dimensional spatial patterns before clustering. The retained component count is chosen to explain at least 85 percent of the standardized feature variance by default.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell fits PCA on X_scaled and chooses the smallest number of components
# whose cumulative explained variance reaches PCA_VARIANCE_TARGET.
# X_pca is used for clustering; X_pca_2d is only for visualization.
# --- End learning comments ---

max_pca_components = min(X_scaled.shape[0], X_scaled.shape[1])
pca_full = PCA(n_components=max_pca_components, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

pca_variance_table = pd.DataFrame(
    {
        "component": np.arange(1, len(explained_variance) + 1),
        "explained_variance_ratio": explained_variance,
        "cumulative_explained_variance": cumulative_variance,
    }
)
pca_variance_table.to_csv(OUTPUT_DIR / "pca_explained_variance_smoothed.csv", index=False)

pca_components_used = int(np.searchsorted(cumulative_variance, PCA_VARIANCE_TARGET) + 1)
pca_explained_variance_retained = float(cumulative_variance[pca_components_used - 1])

X_pca = X_pca_full[:, :pca_components_used]
if X_pca_full.shape[1] >= 2:
    X_pca_2d = X_pca_full[:, :2]
else:
    X_pca_2d = np.column_stack([X_pca_full[:, 0], np.zeros(X_pca_full.shape[0])])

print(f"PCA components used: {pca_components_used}")
print(f"Cumulative explained variance retained: {pca_explained_variance_retained:.4f}")
display(pca_variance_table.head(15))

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(
    pca_variance_table["component"],
    pca_variance_table["cumulative_explained_variance"],
    marker="o",
    linewidth=2,
)
ax.axhline(PCA_VARIANCE_TARGET, color="red", linestyle="--", label=f"Target = {PCA_VARIANCE_TARGET:.0%}")
ax.axvline(pca_components_used, color="gray", linestyle=":", label=f"Components used = {pca_components_used}")
ax.set_title("Experiment B PCA Cumulative Explained Variance")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pca_variance_plot_smoothed.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved PCA variance table: {OUTPUT_DIR / 'pca_explained_variance_smoothed.csv'}")
print(f"Saved PCA variance plot: {OUTPUT_DIR / 'pca_variance_plot_smoothed.png'}")

## 8. Evaluate KMeans Cluster Numbers

### What this cell does

This cell fits KMeans for several candidate values of `k` and calculates four metrics.

### KMeans objective

KMeans assigns samples to clusters by minimizing within-cluster squared distance:

```text
min sum_i ||x_i - c_cluster(i)||^2
```

### Metrics

- `inertia`: within-cluster sum of squared distances. Lower is better, but it naturally decreases as `k` increases.
- `silhouette_score`: compares within-cluster compactness to nearest-cluster separation. Higher is better.
- `calinski_harabasz_score`: compares between-cluster dispersion to within-cluster dispersion. Higher is better.
- `davies_bouldin_score`: compares cluster similarity and spread. Lower is better.

### Interpretation caution

Do not choose `k` only from one metric. For tactical analysis, centroid heatmaps and representative team-seasons must also make sense.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell evaluates multiple KMeans cluster counts.
# KMeans minimizes inertia: sum_i ||x_i - centroid_cluster(i)||^2.
# The metrics help compare k values, but tactical interpretability still matters.
# --- End learning comments ---

def fit_kmeans_model(matrix, n_clusters):
    # Fit KMeans with modern n_init='auto', falling back for older sklearn versions.
    try:
        model = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init="auto")
        model.fit(matrix)
    except TypeError:
        model = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=20)
        model.fit(matrix)
    return model


metric_rows = []
valid_k_values = [k for k in K_RANGE if 1 < k < len(X_pca)]

if not valid_k_values:
    raise ValueError("No valid k values available. Need more filtered samples for clustering evaluation.")

skipped_k_values = [k for k in K_RANGE if k not in valid_k_values]
if skipped_k_values:
    print(f"Skipped k values because there are not enough samples: {skipped_k_values}")

for k in valid_k_values:
    model = fit_kmeans_model(X_pca, k)
    labels = model.labels_

    metric_rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette_score": silhouette_score(X_pca, labels),
            "calinski_harabasz_score": calinski_harabasz_score(X_pca, labels),
            "davies_bouldin_score": davies_bouldin_score(X_pca, labels),
        }
    )

cluster_metric_scores = pd.DataFrame(metric_rows)
cluster_metric_scores.to_csv(OUTPUT_DIR / "cluster_metric_scores_smoothed.csv", index=False)

display(cluster_metric_scores)

fig, axs = plt.subplots(2, 2, figsize=(13, 9))

axs[0, 0].plot(cluster_metric_scores["k"], cluster_metric_scores["inertia"], marker="o")
axs[0, 0].set_title("Inertia by k")
axs[0, 0].set_xlabel("k")
axs[0, 0].set_ylabel("Inertia")

axs[0, 1].plot(cluster_metric_scores["k"], cluster_metric_scores["silhouette_score"], marker="o")
axs[0, 1].set_title("Silhouette score by k")
axs[0, 1].set_xlabel("k")
axs[0, 1].set_ylabel("Silhouette score")

axs[1, 0].plot(cluster_metric_scores["k"], cluster_metric_scores["calinski_harabasz_score"], marker="o")
axs[1, 0].set_title("Calinski-Harabasz score by k")
axs[1, 0].set_xlabel("k")
axs[1, 0].set_ylabel("Calinski-Harabasz score")

axs[1, 1].plot(cluster_metric_scores["k"], cluster_metric_scores["davies_bouldin_score"], marker="o")
axs[1, 1].set_title("Davies-Bouldin score by k")
axs[1, 1].set_xlabel("k")
axs[1, 1].set_ylabel("Davies-Bouldin score")

for ax in axs.flat:
    ax.grid(alpha=0.25)
    ax.axvline(FINAL_K, color="gray", linestyle=":", alpha=0.75)

fig.suptitle("Experiment B - KMeans Cluster Metric Scores", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cluster_metric_scores_smoothed.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved cluster metric scores: {OUTPUT_DIR / 'cluster_metric_scores_smoothed.csv'}")
print(f"Saved cluster metric plot: {OUTPUT_DIR / 'cluster_metric_scores_smoothed.png'}")

## 9. Fit Final KMeans Model

### What this cell does

This cell fits the final KMeans model using `FINAL_K` and adds the resulting cluster label to each filtered team-season.

### Output label

The cluster label column is:

```text
cluster_smoothed
```

### Important detail

The model is fitted in PCA space, not the original 192-dimensional feature space. Later interpretation returns to the original zone features by averaging the original distributions within each cluster.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell fits the final KMeans model in PCA space.
# The resulting integer label is stored as cluster_smoothed and joined back to metadata.
# --- End learning comments ---

if FINAL_K >= len(X_pca):
    raise ValueError(f"FINAL_K={FINAL_K} is not valid for {len(X_pca)} samples.")

final_kmeans = fit_kmeans_model(X_pca, FINAL_K)
cluster_smoothed = final_kmeans.labels_

clustered_team_seasons = filtered.reset_index(drop=True).copy()
clustered_team_seasons["cluster_smoothed"] = cluster_smoothed
clustered_team_seasons["pca_1"] = X_pca_2d[:, 0]
clustered_team_seasons["pca_2"] = X_pca_2d[:, 1]

cluster_counts = clustered_team_seasons["cluster_smoothed"].value_counts().sort_index()
cluster_percentages = (cluster_counts / len(clustered_team_seasons) * 100).round(2)

cluster_size_table = pd.DataFrame(
    {
        "cluster_smoothed": cluster_counts.index,
        "n_samples": cluster_counts.values,
        "sample_share_percent": cluster_percentages.values,
    }
)

clustered_team_seasons.to_csv(OUTPUT_DIR / "clustered_team_seasons_smoothed.csv", index=False)

display(cluster_size_table)
print(f"Saved clustered dataset: {OUTPUT_DIR / 'clustered_team_seasons_smoothed.csv'}")

## 10. PCA Scatter Visualization

### What this cell does

This cell plots team-seasons in the first two PCA dimensions and colors each point by `cluster_smoothed`.

### Why this plot is useful

The 2D scatter helps you see whether clusters are visually separated in the strongest PCA directions.

### Limitation

Only PC1 and PC2 are shown. The final KMeans model may use many PCA components, so the scatter plot is a simplified view. It should support interpretation, not replace centroid heatmaps.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell visualizes the clusters using PC1 and PC2.
# The labeled version annotates only representative points to avoid visual clutter.
# --- End learning comments ---

def label_for_row(row):
    parts = []
    if "team_name" in row and pd.notna(row["team_name"]):
        parts.append(str(row["team_name"]))
    if "season_name" in row and pd.notna(row["season_name"]):
        parts.append(str(row["season_name"]))
    elif "season_id" in row and pd.notna(row["season_id"]):
        parts.append(str(row["season_id"]))
    return " - ".join(parts) if parts else str(row.name)


def plot_pca_scatter(frame, output_path, label_indices=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(
        frame["pca_1"],
        frame["pca_2"],
        c=frame["cluster_smoothed"],
        cmap="tab10",
        alpha=0.78,
        s=45,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title("Experiment B - Smoothed xT Matrix Clusters")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(alpha=0.25)
    legend = ax.legend(*scatter.legend_elements(), title="Cluster", loc="best")
    ax.add_artist(legend)

    if label_indices is not None:
        for idx in label_indices:
            row = frame.loc[idx]
            ax.annotate(
                label_for_row(row),
                (row["pca_1"], row["pca_2"]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
                alpha=0.85,
            )

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


plot_pca_scatter(clustered_team_seasons, OUTPUT_DIR / "pca_scatter_clusters_smoothed.png")

assigned_centroid_indices, assigned_distances = pairwise_distances_argmin_min(X_pca, final_kmeans.cluster_centers_)
clustered_team_seasons["distance_to_centroid"] = assigned_distances

label_indices = []
for cluster_id in sorted(clustered_team_seasons["cluster_smoothed"].unique()):
    cluster_subset = clustered_team_seasons.loc[clustered_team_seasons["cluster_smoothed"] == cluster_id]
    label_indices.extend(cluster_subset.nsmallest(2, "distance_to_centroid").index.tolist())

plot_pca_scatter(
    clustered_team_seasons,
    OUTPUT_DIR / "pca_scatter_clusters_smoothed_labeled.png",
    label_indices=label_indices,
)

print(f"Saved PCA scatter plot: {OUTPUT_DIR / 'pca_scatter_clusters_smoothed.png'}")
print(f"Saved labeled PCA scatter plot: {OUTPUT_DIR / 'pca_scatter_clusters_smoothed_labeled.png'}")

## 11. Cluster Summary

### What this cell does

This cell creates a summary table for each cluster using available metadata and optional interpretation columns.

### Aggregation logic

For each cluster:

- count team-seasons
- calculate sample share
- list common competitions and teams
- average available numeric columns such as `match_count`, `total_positive_xT`, and spatial share features

### Why this matters

This table helps you understand whether a cluster is dominated by certain competitions, teams, eras, or broad spatial tendencies.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell summarizes each cluster with counts, shares, common categories,
# and means of any optional numeric interpretation columns that are available.
# --- End learning comments ---

def most_common_values(series, top_n=3):
    counts = series.dropna().astype(str).value_counts().head(top_n)
    if counts.empty:
        return ""
    return "; ".join(f"{name} ({count})" for name, count in counts.items())


summary_numeric_columns = [
    "match_count",
    "move_action_count",
    "total_positive_xT",
    "attacking_third_share",
    "middle_third_share",
    "defensive_third_share",
    "left_side_share",
    "center_share",
    "right_side_share",
    "pass_xT_share",
    "carry_xT_share",
]

cluster_summary_rows = []
for cluster_id, group in clustered_team_seasons.groupby("cluster_smoothed"):
    row = {
        "cluster_smoothed": cluster_id,
        "n_team_seasons": len(group),
        "share_of_total_samples": len(group) / len(clustered_team_seasons),
    }

    if "competition_name" in group.columns:
        row["most_common_competitions"] = most_common_values(group["competition_name"])
    if "team_name" in group.columns:
        row["most_common_teams"] = most_common_values(group["team_name"])

    for column in summary_numeric_columns:
        if column in group.columns:
            row[f"avg_{column}"] = pd.to_numeric(group[column], errors="coerce").mean()

    cluster_summary_rows.append(row)

cluster_summary = pd.DataFrame(cluster_summary_rows).sort_values("cluster_smoothed").reset_index(drop=True)
cluster_summary.to_csv(OUTPUT_DIR / "cluster_summary_smoothed.csv", index=False)

display(cluster_summary)
print(f"Saved cluster summary: {OUTPUT_DIR / 'cluster_summary_smoothed.csv'}")

## 12. Cluster Centroid Analysis

Centroid heatmaps show the average smoothed xT creation spatial profile of each cluster and are the main tool for interpreting clusters tactically. Even though KMeans is fit in PCA space, the centroids below are calculated as the mean of the original unscaled 192 smoothed xT distribution features within each cluster.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell interprets clusters in the original 192-zone feature space.
# For each cluster, centroid_j = mean of original unscaled feature j among cluster members.
# The centroid vector is reshaped to a 12 by 16 grid for pitch heatmaps.
# --- End learning comments ---

cluster_centroids_smoothed = (
    clustered_team_seasons.groupby("cluster_smoothed")[feature_columns]
    .mean()
    .sort_index()
)
cluster_centroids_smoothed.to_csv(OUTPUT_DIR / "cluster_centroids_smoothed.csv")

centroid_grids = {
    cluster_id: row.values.reshape(GRID_W, GRID_L)
    for cluster_id, row in cluster_centroids_smoothed.iterrows()
}

vmax = max(float(grid.max()) for grid in centroid_grids.values())
vmin = 0.0

n_clusters = len(cluster_centroids_smoothed)
ncols = min(3, n_clusters)
nrows = math.ceil(n_clusters / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5.2 * ncols, 3.8 * nrows))
axes = np.asarray(axes).reshape(-1)

for ax, (cluster_id, grid) in zip(axes, centroid_grids.items()):
    image = ax.imshow(grid, origin="lower", aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"Cluster {cluster_id}")
    ax.set_xlabel("Pitch length bin")
    ax.set_ylabel("Pitch width bin")
    ax.set_xticks(range(GRID_L))
    ax.set_yticks(range(GRID_W))
    ax.tick_params(labelsize=7)

for ax in axes[len(centroid_grids):]:
    ax.axis("off")

fig.suptitle("Experiment B - Cluster Centroid Heatmaps", fontsize=16, fontweight="bold")
fig.colorbar(image, ax=axes[:len(centroid_grids)], shrink=0.78, label="Average xT share")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cluster_centroid_heatmaps_smoothed.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved cluster centroid vectors: {OUTPUT_DIR / 'cluster_centroids_smoothed.csv'}")
print(f"Saved cluster centroid heatmaps: {OUTPUT_DIR / 'cluster_centroid_heatmaps_smoothed.png'}")

## 13. Top Zones Per Cluster

### What this cell does

This cell identifies the 10 highest-value zones in each cluster centroid.

### Formula

For each cluster centroid vector:

```text
top zones = argsort(centroid values, descending)[:10]
```

Each selected zone is converted back into:

```text
x_bin = zone % 16
y_bin = zone // 16
```

### Why this helps

Top zones provide a compact numeric summary of each centroid heatmap.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell ranks each cluster centroid's zones from highest to lowest average xT share.
# It converts zone id back to grid coordinates with x_bin = zone % 16 and y_bin = zone // 16.
# --- End learning comments ---

top_zone_rows = []

for cluster_id, centroid in cluster_centroids_smoothed.iterrows():
    top_columns = centroid.sort_values(ascending=False).head(10)
    for rank, (column, value) in enumerate(top_columns.items(), start=1):
        zone = zone_number_from_column(column, FEATURE_PREFIX_USED)
        top_zone_rows.append(
            {
                "cluster_smoothed": cluster_id,
                "rank": rank,
                "zone": zone,
                "x_bin": zone % GRID_L,
                "y_bin": zone // GRID_L,
                "centroid_value": value,
            }
        )

cluster_top_zones = pd.DataFrame(top_zone_rows)
cluster_top_zones.to_csv(OUTPUT_DIR / "cluster_top_zones_smoothed.csv", index=False)

display(cluster_top_zones)
print(f"Saved cluster top zones: {OUTPUT_DIR / 'cluster_top_zones_smoothed.csv'}")

## 14. Representative Team-Seasons

Representative team-seasons are the samples closest to their assigned KMeans centroid in PCA clustering space. They are useful examples for manual cluster naming, but they should not be treated as ground-truth labels.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell finds representative team-seasons nearest to each cluster centroid.
# Distance is Euclidean distance in PCA space: ||sample - assigned_centroid||.
# --- End learning comments ---

representative_columns = [
    "cluster_smoothed",
    "team_name",
    "season_name",
    "competition_name",
    "distance_to_centroid",
    "total_positive_xT",
    "match_count",
    "move_action_count",
]
available_representative_columns = [
    column for column in representative_columns if column in clustered_team_seasons.columns
]

representatives = (
    clustered_team_seasons.sort_values(["cluster_smoothed", "distance_to_centroid"])
    .groupby("cluster_smoothed", as_index=False)
    .head(5)
    .loc[:, available_representative_columns]
    .reset_index(drop=True)
)

representatives.to_csv(OUTPUT_DIR / "cluster_representatives_smoothed.csv", index=False)

display(representatives)
print(f"Saved representative team-seasons: {OUTPUT_DIR / 'cluster_representatives_smoothed.csv'}")

## 15. Cluster Naming Support Table

This support table gives neutral evidence for manual cluster names. The tentative descriptions are based only on observed smoothed xT spatial features and avoid strong labels such as counterattack, high pressing, or long-ball unless the available features directly support them.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell builds neutral naming support from centroid-derived thirds, width channels,
# top zones, and representative examples. It avoids overclaiming tactical labels.
# --- End learning comments ---

left_zone_columns = [
    column for column in feature_columns
    if (zone_number_from_column(column, FEATURE_PREFIX_USED) // GRID_L) in range(0, 4)
]
center_zone_columns = [
    column for column in feature_columns
    if (zone_number_from_column(column, FEATURE_PREFIX_USED) // GRID_L) in range(4, 8)
]
right_zone_columns = [
    column for column in feature_columns
    if (zone_number_from_column(column, FEATURE_PREFIX_USED) // GRID_L) in range(8, 12)
]
defensive_third_columns = [
    column for column in feature_columns
    if (zone_number_from_column(column, FEATURE_PREFIX_USED) % GRID_L) in range(0, 5)
]
middle_third_columns = [
    column for column in feature_columns
    if (zone_number_from_column(column, FEATURE_PREFIX_USED) % GRID_L) in range(5, 11)
]
attacking_third_columns = [
    column for column in feature_columns
    if (zone_number_from_column(column, FEATURE_PREFIX_USED) % GRID_L) in range(11, 16)
]


def centroid_share(row, columns):
    return float(row[columns].sum())


def neutral_description(row):
    attacking = row.get("attacking_third_share", np.nan)
    middle = row.get("middle_third_share", np.nan)
    defensive = row.get("defensive_third_share", np.nan)
    left = row.get("left_side_share", np.nan)
    center = row.get("center_share", np.nan)
    right = row.get("right_side_share", np.nan)

    if pd.notna(attacking) and attacking > max(middle, defensive) + 0.05:
        return "High attacking-third xT concentration"
    if pd.notna(defensive) and defensive > max(middle, attacking) + 0.05:
        return "Deep-origin xT creation profile"
    if pd.notna(center) and center > max(left, right) + 0.05:
        return "Central xT creation profile"
    if pd.notna(left) and pd.notna(right) and max(left, right) > center + 0.05:
        return "Wide-side xT creation profile"
    return "Balanced xT creation distribution"


naming_rows = []

for cluster_id, centroid in cluster_centroids_smoothed.iterrows():
    top_three = centroid.sort_values(ascending=False).head(3)
    top_zones = [
        (zone_number_from_column(column, FEATURE_PREFIX_USED), value)
        for column, value in top_three.items()
    ]

    row = {
        "cluster_smoothed": cluster_id,
        "n_team_seasons": int(cluster_counts.loc[cluster_id]),
        "top_zone_1": top_zones[0][0],
        "top_zone_1_share": top_zones[0][1],
        "top_zone_2": top_zones[1][0],
        "top_zone_2_share": top_zones[1][1],
        "top_zone_3": top_zones[2][0],
        "top_zone_3_share": top_zones[2][1],
        "attacking_third_share": centroid_share(centroid, attacking_third_columns),
        "middle_third_share": centroid_share(centroid, middle_third_columns),
        "defensive_third_share": centroid_share(centroid, defensive_third_columns),
        "left_side_share": centroid_share(centroid, left_zone_columns),
        "center_share": centroid_share(centroid, center_zone_columns),
        "right_side_share": centroid_share(centroid, right_zone_columns),
    }

    cluster_reps = representatives.loc[representatives["cluster_smoothed"] == cluster_id].head(3)
    if "team_name" in cluster_reps.columns:
        if "season_name" in cluster_reps.columns:
            row["example_representative_teams"] = "; ".join(
                f"{team} ({season})"
                for team, season in zip(cluster_reps["team_name"], cluster_reps["season_name"])
            )
        else:
            row["example_representative_teams"] = "; ".join(cluster_reps["team_name"].astype(str))
    else:
        row["example_representative_teams"] = ""

    row["tentative_description"] = neutral_description(row)
    naming_rows.append(row)

cluster_naming_support = pd.DataFrame(naming_rows)
ordered_naming_columns = [
    "cluster_smoothed",
    "n_team_seasons",
    "tentative_description",
    "top_zone_1",
    "top_zone_1_share",
    "top_zone_2",
    "top_zone_2_share",
    "top_zone_3",
    "top_zone_3_share",
    "attacking_third_share",
    "middle_third_share",
    "defensive_third_share",
    "left_side_share",
    "center_share",
    "right_side_share",
    "example_representative_teams",
]
cluster_naming_support = cluster_naming_support[ordered_naming_columns]
cluster_naming_support.to_csv(OUTPUT_DIR / "cluster_naming_support_smoothed.csv", index=False)

display(cluster_naming_support)
print(f"Saved cluster naming support table: {OUTPUT_DIR / 'cluster_naming_support_smoothed.csv'}")

## 16. Optional Comparison With Experiment A

If Experiment A raw clustering outputs exist, this section compares raw and smoothed cluster assignments. KMeans cluster IDs are arbitrary, so raw cluster 0 should not be interpreted as the same kind of cluster as smoothed cluster 0. Adjusted Rand Index and Normalized Mutual Information provide label-invariant comparison.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This optional section compares Experiment B with Experiment A only when A outputs exist.
# ARI and NMI compare cluster structures without assuming that numeric cluster IDs match.
# --- End learning comments ---

compared_with_experiment_A = False
experiment_A_overlap_n = np.nan
ARI_vs_experiment_A = np.nan
NMI_vs_experiment_A = np.nan

if EXPERIMENT_A_CLUSTERED_FILE.exists():
    experiment_A_clustered = pd.read_csv(EXPERIMENT_A_CLUSTERED_FILE)

    id_merge_keys = ["competition_id", "season_id", "team_id"]
    name_merge_keys = ["competition_name", "season_name", "team_name"]
    if all(column in experiment_A_clustered.columns and column in clustered_team_seasons.columns for column in id_merge_keys):
        merge_keys = id_merge_keys
    elif all(column in experiment_A_clustered.columns and column in clustered_team_seasons.columns for column in name_merge_keys):
        merge_keys = name_merge_keys
    else:
        merge_keys = []

    if not merge_keys:
        print("Experiment A file exists, but no stable merge keys are available. Skipping assignment comparison.")
    elif "cluster_raw" not in experiment_A_clustered.columns:
        print("Experiment A file exists, but it does not contain cluster_raw. Skipping assignment comparison.")
    else:
        comparison = experiment_A_clustered[merge_keys + ["cluster_raw"]].merge(
            clustered_team_seasons[merge_keys + ["cluster_smoothed"]],
            on=merge_keys,
            how="inner",
        )
        experiment_A_overlap_n = int(len(comparison))
        print(f"Overlapping team-seasons: {experiment_A_overlap_n:,}")

        if experiment_A_overlap_n > 0:
            compared_with_experiment_A = True
            ARI_vs_experiment_A = adjusted_rand_score(comparison["cluster_raw"], comparison["cluster_smoothed"])
            NMI_vs_experiment_A = normalized_mutual_info_score(comparison["cluster_raw"], comparison["cluster_smoothed"])

            overlap_counts = pd.crosstab(
                comparison["cluster_raw"],
                comparison["cluster_smoothed"],
                rownames=["cluster_raw"],
                colnames=["cluster_smoothed"],
            )
            overlap_row_normalized = overlap_counts.div(overlap_counts.sum(axis=1), axis=0)
            comparison_metrics = pd.DataFrame([{"overlap_n": experiment_A_overlap_n, "adjusted_rand_index": ARI_vs_experiment_A, "normalized_mutual_information": NMI_vs_experiment_A}])

            overlap_counts.to_csv(OUTPUT_DIR / "comparison_with_experiment_A_cluster_overlap.csv")
            overlap_row_normalized.to_csv(OUTPUT_DIR / "comparison_with_experiment_A_cluster_overlap_row_normalized.csv")
            comparison_metrics.to_csv(OUTPUT_DIR / "comparison_with_experiment_A_metrics.csv", index=False)

            print(f"Adjusted Rand Index: {ARI_vs_experiment_A:.4f}")
            print(f"Normalized Mutual Information: {NMI_vs_experiment_A:.4f}")
            display(comparison_metrics)
            display(overlap_counts)
            display(overlap_row_normalized)
        else:
            print("Experiment A file exists, but no overlapping team-seasons were found after merging.")
else:
    print(f"Experiment A clustered file not found: {EXPERIMENT_A_CLUSTERED_FILE}. Skipping assignment comparison.")

if EXPERIMENT_A_SUMMARY_FILE.exists():
    experiment_A_summary = pd.read_csv(EXPERIMENT_A_SUMMARY_FILE)
    experiment_A_summary.insert(0, "experiment", "Experiment A - Raw")
    experiment_B_summary = cluster_summary.copy()
    experiment_B_summary.insert(0, "experiment", "Experiment B - Smoothed")
    comparison_summary = pd.concat([experiment_A_summary, experiment_B_summary], ignore_index=True, sort=False)
    comparison_summary.to_csv(OUTPUT_DIR / "comparison_with_experiment_A_summary.csv", index=False)
    display(comparison_summary)
    print(f"Saved side-by-side summary: {OUTPUT_DIR / 'comparison_with_experiment_A_summary.csv'}")
else:
    print(f"Experiment A summary file not found: {EXPERIMENT_A_SUMMARY_FILE}. Skipping summary comparison.")

if EXPERIMENT_A_REPRESENTATIVES_FILE.exists():
    experiment_A_representatives = pd.read_csv(EXPERIMENT_A_REPRESENTATIVES_FILE)
    experiment_A_representatives.insert(0, "experiment", "Experiment A - Raw")
    experiment_B_representatives = representatives.copy()
    experiment_B_representatives.insert(0, "experiment", "Experiment B - Smoothed")
    comparison_representatives = pd.concat([experiment_A_representatives, experiment_B_representatives], ignore_index=True, sort=False)
    comparison_representatives.to_csv(OUTPUT_DIR / "comparison_with_experiment_A_representatives.csv", index=False)
    display(comparison_representatives)
    print(f"Saved representative comparison: {OUTPUT_DIR / 'comparison_with_experiment_A_representatives.csv'}")
else:
    print(f"Experiment A representatives file not found: {EXPERIMENT_A_REPRESENTATIVES_FILE}. Skipping representative comparison.")


### Interpreting Experiment A Comparison

High ARI or NMI suggests smoothing preserves the main grouping structure from the raw matrix. Low ARI or NMI suggests smoothing meaningfully changes the cluster structure. Neither outcome is automatically better. The final decision should consider centroid heatmap clarity, top-zone coherence, and whether representative team-seasons make tactical sense.

This comparison does not load or process `cluster_centroid_heatmaps_raw.png`; that file is only a visual reference from Experiment A and is not used as data.


## 17. Experiment Summary

This one-row summary captures the final sample, feature count, PCA settings, clustering scores, and optional Experiment A comparison metrics. It is designed to make Experiment B easy to compare with later experiments.


In [ ]:
# --- Learning comments added by notebook-learning-commentator ---
# This cell saves a one-row record of the final experiment settings and metrics.
# It is designed for comparing Experiment B against later clustering experiments.
# --- End learning comments ---

final_silhouette = silhouette_score(X_pca, cluster_smoothed)
final_calinski_harabasz = calinski_harabasz_score(X_pca, cluster_smoothed)
final_davies_bouldin = davies_bouldin_score(X_pca, cluster_smoothed)

experiment_summary = pd.DataFrame(
    [
        {
            "experiment_name": EXPERIMENT_NAME,
            "input_file": str(INPUT_FILE),
            "output_folder": str(OUTPUT_DIR),
            "number_of_samples": len(clustered_team_seasons),
            "number_of_features": len(feature_columns),
            "final_k": FINAL_K,
            "pca_components_used": pca_components_used,
            "pca_explained_variance_retained": pca_explained_variance_retained,
            "final_silhouette_score": final_silhouette,
            "final_calinski_harabasz_score": final_calinski_harabasz,
            "final_davies_bouldin_score": final_davies_bouldin,
            "feature_prefix_used": FEATURE_PREFIX_USED,
            "compared_with_experiment_A": compared_with_experiment_A,
            "experiment_A_overlap_n": experiment_A_overlap_n,
            "ARI_vs_experiment_A": ARI_vs_experiment_A,
            "NMI_vs_experiment_A": NMI_vs_experiment_A,
        }
    ]
)
experiment_summary.to_csv(OUTPUT_DIR / "experiment_summary_smoothed.csv", index=False)

display(experiment_summary)
print(f"Saved experiment summary: {OUTPUT_DIR / 'experiment_summary_smoothed.csv'}")

## Experiment B Interpretation Notes

Smoothed xT matrix clustering captures where team-seasons tend to create possession threat after isolated zone spikes have been softened. This can make centroid heatmaps easier to read because they emphasize broader spatial profiles rather than one-off high-value cells.

The method may miss tactical behaviors that are not represented in possession xT creation. It does not directly measure pressing, defensive block height, counterpressing, build-up speed, opponent context, or whether similar xT zones came from different action types.

Whether smoothing reduces noise should be judged from the centroid heatmaps, top-zone tables, representative team-seasons, and optional comparison with Experiment A. If the smoothed centroids are clearer while the representatives remain coherent, smoothing may improve robustness. If the smoothed clusters collapse distinct raw patterns into vague averages, smoothing may be too strong or less informative for this task.

If Experiment A comparison outputs are available, ARI and NMI show whether the raw and smoothed cluster structures are similar. A high value suggests the main grouping structure is preserved after smoothing. A low value suggests smoothing changes the grouping structure. Neither result is automatically better.

Centroid heatmaps are more useful than PCA scatter plots for tactical interpretation because the heatmaps return the cluster profile to the original 16 by 12 pitch grid. PCA scatter plots are helpful diagnostics, but the first two components may not preserve all meaningful spatial differences.

Clusters should remain hypotheses rather than final tactical labels. Experiment A is the raw baseline. Experiment B tests whether spatial smoothing improves robustness. Experiment C should test whether multi-channel features provide richer tactical interpretation than a single smoothed xT created distribution.
